# 🏥 IGD Smart Queue System
### Optimasi Antrean IGD dengan Hash Table & Priority Queue

---
**Smart-Triage Team | Struktur Data 2024**  
> *"Efisiensi waktu adalah kunci utama dalam menyelamatkan nyawa di instalasi gawat darurat."*

---
## 📌 1. Latar Belakang

### Mengapa sistem antrean tradisional sering gagal menyelamatkan nyawa?

Bayangkan kamu pergi ke IGD rumah sakit dengan kondisi darurat — jantung berdenyut sangat cepat, SpO₂ di bawah 90%. Tapi karena kamu datang belakangan, kamu harus antre di belakang pasien yang hanya mengalami luka ringan.

Inilah bahaya sistem **First-In First-Out (FIFO)** di lingkungan IGD:

| Kelemahan FIFO | Dampak |
|---|---|
| Pasien kritis bisa tertahan di belakang antrean | Risiko kematian meningkat |
| Administrasi manual (cari berkas fisik) | Memperlambat tindakan medis darurat |
| Tidak bisa merespons perubahan kondisi pasien | Sistem kaku dan tidak adaptif |

### 💡 Solusi: Smart-Triage

Sistem **Smart-Triage** menggabungkan dua struktur data:
- **Hash Table** → sebagai *"Database Instan"* untuk mengakses riwayat medis pasien dalam waktu O(1)
- **Priority Queue** → sebagai *"Saringan Pintar"* yang otomatis mengurutkan antrean berdasarkan skor urgensi medis

---
## 🎯 2. Tujuan Proyek

1. **Mensimulasikan** sistem antrean IGD berbasis prioritas medis
2. **Mengimplementasikan** Hash Table (`dict`) untuk manajemen data pasien
3. **Mengimplementasikan** Priority Queue (`heapq`) untuk penentuan urutan pelayanan
4. **Membuktikan** bahwa pasien kritis yang datang belakangan tetap bisa dilayani lebih dulu
5. **Memahami** keunggulan algoritma O(log n) dibanding sorting manual O(n log n)

---
## 📚 3. Konsep Data Structure

### 3.1 Hash Table — *"Lemari Arsip Super Cepat"*

Bayangkan kamu punya lemari arsip dengan 1.000 map pasien. Cara lama: buka satu per satu sampai ketemu. Cara Hash Table: setiap map punya **kode unik (key)** — langsung ambil tanpa cari-cari.

```
hash_table["P001"]  →  langsung dapat data Andi
hash_table["P002"]  →  langsung dapat data Budi
```

**Kompleksitas: O(1)** — konstan, tidak peduli ada 10 atau 10.000 pasien.

Dalam Python, Hash Table diimplementasikan dengan **`dict`**.

---

### 3.2 Priority Queue (Heap) — *"Antrean dengan Jalur Prioritas"*

Bayangkan antrean biasa seperti antrean bioskop — siapa duluan dilayani duluan. Priority Queue seperti antrean di IGD yang **punya sistem jalur darurat**: pasien paling kritis selalu berada di posisi terdepan, berapapun urutan kedatangannya.

```
Antrean biasa:  Siti(50) → Andi(20)
Budi(95) datang → Priority Queue: Budi(95) → Siti(50) → Andi(20)
```

**Kompleksitas insert/delete: O(log n)** — jauh lebih cepat dari sorting manual O(n log n).

Dalam Python, Priority Queue diimplementasikan dengan modul **`heapq`**.

> ⚠️ **Catatan:** Python `heapq` adalah **min-heap** (nilai terkecil di atas). Karena kita butuh nilai terbesar (skor tertinggi = paling darurat) di depan, kita gunakan **nilai negatif** sebagai trik.

---

### 3.3 Logika Skor Urgensi

Setiap pasien mendapat skor berdasarkan kondisi medis:

| Parameter Medis | Kondisi Pasien | Tambahan Poin |
|---|---|---|
| Kesadaran | Pingsan / Tidak Sadar | +50 Poin |
| Oksigen (SpO₂) | Di bawah 90% | +40 Poin |
| Detak Jantung | Sangat Cepat (>120 bpm) | +30 Poin |
| Tingkat Nyeri | Nyeri Berat (Skala 8–10) | +20 Poin |

> Total skor inilah yang menentukan posisi pasien di dalam antrean prioritas.

---
## 📦 4. Import Library

In [ ]:
# ============================================================
# IMPORT LIBRARY
# Kita hanya butuh 1 library bawaan Python: heapq
# Tidak ada AI, tidak ada sklearn, tidak ada tensorflow.
# Murni data structure!
# ============================================================

import heapq  # Untuk Priority Queue (min-heap)

print("✅ Library berhasil diimport!")
print("📌 heapq  → Priority Queue (urutan prioritas)")
print("📌 dict   → Hash Table (akses data O(1)) — sudah built-in Python!")

---
## 🗂️ 5. Dataset Pasien

Kita akan mensimulasikan 6 pasien IGD dengan kondisi medis yang berbeda-beda.

Setiap pasien memiliki:
- `id` → ID unik pasien (digunakan sebagai **key** di Hash Table)
- `nama` → nama pasien
- `kondisi` → deskripsi kondisi medis
- `kesadaran` → apakah pingsan/tidak sadar?
- `spo2` → kadar oksigen dalam darah (%)
- `detak_jantung` → detak jantung (bpm)
- `nyeri` → skala nyeri 1–10

In [ ]:
# ============================================================
# DATASET PASIEN IGD
# Data ini merepresentasikan pasien yang datang ke IGD
# dengan waktu kedatangan dan kondisi berbeda-beda.
# ============================================================

# List pasien yang akan datang secara berurutan
# (urutan list = urutan kedatangan ke IGD)
data_pasien = [
    {
        "id": "P001",
        "nama": "Andi",
        "kondisi": "Luka ringan pada tangan",
        "kesadaran": True,   # Sadar
        "spo2": 98,          # Normal
        "detak_jantung": 80, # Normal
        "nyeri": 3           # Nyeri ringan
    },
    {
        "id": "P002",
        "nama": "Siti",
        "kondisi": "Demam tinggi dan nyeri kepala",
        "kesadaran": True,   # Sadar
        "spo2": 95,          # Sedikit turun
        "detak_jantung": 100,# Sedikit cepat
        "nyeri": 6           # Nyeri sedang
    },
    {
        "id": "P003",
        "nama": "Budi",
        "kondisi": "KRITIS: Tidak sadarkan diri, SpO2 rendah",
        "kesadaran": False,  # ⚠️ TIDAK SADAR
        "spo2": 82,          # ⚠️ DI BAWAH 90%
        "detak_jantung": 130,# ⚠️ SANGAT CEPAT
        "nyeri": 9           # ⚠️ NYERI BERAT
    },
    {
        "id": "P004",
        "nama": "Dewi",
        "kondisi": "Sesak napas ringan",
        "kesadaran": True,   # Sadar
        "spo2": 92,          # Sedikit di bawah normal
        "detak_jantung": 95, # Normal
        "nyeri": 5           # Nyeri sedang
    },
    {
        "id": "P005",
        "nama": "Reza",
        "kondisi": "KRITIS: Detak jantung sangat cepat, nyeri dada hebat",
        "kesadaran": True,   # Masih sadar
        "spo2": 87,          # ⚠️ DI BAWAH 90%
        "detak_jantung": 145,# ⚠️ SANGAT CEPAT
        "nyeri": 9           # ⚠️ NYERI BERAT
    },
    {
        "id": "P006",
        "nama": "Maya",
        "kondisi": "Keseleo kaki ringan",
        "kesadaran": True,   # Sadar
        "spo2": 99,          # Normal
        "detak_jantung": 75, # Normal
        "nyeri": 4           # Nyeri ringan
    }
]

print(f"✅ Dataset berhasil dimuat: {len(data_pasien)} pasien")
print("\n📋 Daftar pasien (urutan kedatangan ke IGD):")
print("-" * 50)
for i, p in enumerate(data_pasien, 1):
    print(f"  {i}. [{p['id']}] {p['nama']} — {p['kondisi'][:40]}..." 
          if len(p['kondisi']) > 40 else f"  {i}. [{p['id']}] {p['nama']} — {p['kondisi']}")

---
## 🗄️ 6. Hash Table Implementation

### Cara kerja Hash Table di sini:

```
Key (ID Pasien)  →  Value (Data Lengkap Pasien)
─────────────────────────────────────────────
"P001"           →  {nama: "Andi", spo2: 98, ...}
"P002"           →  {nama: "Siti", spo2: 95, ...}
"P003"           →  {nama: "Budi", spo2: 82, ...}
```

Petugas IGD cukup scan kartu identitas pasien → ID masuk → **langsung muncul semua riwayat medis** tanpa harus membuka tumpukan berkas fisik. Ini yang disebut efisiensi **O(1)**.

In [ ]:
# ============================================================
# HASH TABLE IMPLEMENTATION
# Menggunakan dict Python sebagai Hash Table
# Key   = ID Pasien (misal: "P001")
# Value = Seluruh data medis pasien
# ============================================================

# Inisialisasi Hash Table kosong
hash_table_pasien = {}  # dict kosong = Hash Table kosong


def tambah_ke_hash_table(hash_table, pasien):
    """
    Fungsi untuk mendaftarkan pasien ke Hash Table.
    Kompleksitas: O(1) — langsung simpan tanpa perlu cari lokasi.
    """
    patient_id = pasien["id"]          # ambil ID sebagai key
    hash_table[patient_id] = pasien    # simpan seluruh data sebagai value
    print(f"  📥 [{patient_id}] {pasien['nama']} → berhasil didaftarkan ke Hash Table")


def cari_pasien(hash_table, patient_id):
    """
    Fungsi untuk mencari data pasien berdasarkan ID.
    Kompleksitas: O(1) — akses langsung, secepat apapun ukuran database.
    """
    if patient_id in hash_table:       # cek apakah ID ada
        return hash_table[patient_id]  # langsung kembalikan data
    else:
        return None                    # tidak ditemukan


# --- Daftarkan semua pasien ke Hash Table ---
print("🏥 Mendaftarkan pasien ke Hash Table...")
print("-" * 50)
for pasien in data_pasien:
    tambah_ke_hash_table(hash_table_pasien, pasien)

print(f"\n✅ Total pasien terdaftar: {len(hash_table_pasien)} pasien")

# --- Demo: Akses data pasien secara instan ---
print("\n" + "=" * 50)
print("🔍 DEMO: Akses data pasien secara instan (O(1))")
print("=" * 50)

# Misalnya dokter ingin akses data P003
target_id = "P003"
hasil = cari_pasien(hash_table_pasien, target_id)

if hasil:
    print(f"\n📂 Data pasien ID '{target_id}' ditemukan:")
    print(f"   Nama          : {hasil['nama']}")
    print(f"   Kondisi       : {hasil['kondisi']}")
    print(f"   SpO2          : {hasil['spo2']}%")
    print(f"   Detak Jantung : {hasil['detak_jantung']} bpm")
    print(f"   Skala Nyeri   : {hasil['nyeri']}/10")
    print(f"   Kesadaran     : {'Sadar' if hasil['kesadaran'] else '⚠️ TIDAK SADAR'}")

---
## 🧮 7. Logika Penghitungan Skor Urgensi

Sebelum masuk antrean Priority Queue, setiap pasien dihitung dulu skor urgensinya.

Rumus sederhana dari slide Smart-Triage:

```
skor = 0
jika pingsan/tidak sadar   → +50
jika SpO2 < 90%            → +40
jika detak jantung > 120   → +30
jika skala nyeri >= 8      → +20
```

Skor inilah yang akan menentukan siapa yang dilayani lebih dulu.

In [ ]:
# ============================================================
# FUNGSI PENGHITUNGAN SKOR URGENSI
# Berdasarkan tabel dari slide "Logika Penentuan Skor Urgensi"
# Tidak ada AI — ini adalah aturan berbasis threshold medis.
# ============================================================

def hitung_skor_urgensi(pasien):
    """
    Menghitung skor urgensi pasien berdasarkan parameter medis.
    
    Parameter:
        pasien (dict): data pasien dari Hash Table
    
    Returns:
        int: total skor urgensi (semakin tinggi = semakin darurat)
    """
    skor = 0
    rincian = []  # untuk menyimpan penjelasan skor

    # Cek 1: Kesadaran — Pingsan/Tidak Sadar → +50
    if not pasien["kesadaran"]:
        skor += 50
        rincian.append("+50 (tidak sadar)")

    # Cek 2: Oksigen (SpO2) — Di bawah 90% → +40
    if pasien["spo2"] < 90:
        skor += 40
        rincian.append(f"+40 (SpO2={pasien['spo2']}% < 90%)")

    # Cek 3: Detak Jantung — Sangat Cepat (>120 bpm) → +30
    if pasien["detak_jantung"] > 120:
        skor += 30
        rincian.append(f"+30 (detak={pasien['detak_jantung']}bpm > 120)")

    # Cek 4: Tingkat Nyeri — Berat (skala 8-10) → +20
    if pasien["nyeri"] >= 8:
        skor += 20
        rincian.append(f"+20 (nyeri={pasien['nyeri']}/10)")

    return skor, rincian


# --- Hitung dan tampilkan skor semua pasien ---
print("🧮 PERHITUNGAN SKOR URGENSI SEMUA PASIEN")
print("=" * 60)

for pasien in data_pasien:
    skor, rincian = hitung_skor_urgensi(pasien)
    rincian_str = " | ".join(rincian) if rincian else "tidak ada kondisi kritis"
    print(f"\n  [{pasien['id']}] {pasien['nama']}")
    print(f"       Rincian : {rincian_str}")
    print(f"       SKOR    : {skor} poin  {'🔴 KRITIS' if skor >= 70 else '🟡 SEDANG' if skor >= 20 else '🟢 RINGAN'}")

---
## ⚡ 8. Priority Queue Implementation

### Cara kerja `heapq` di Python:

`heapq` adalah **min-heap** — nilai terkecil selalu di atas (root).  
Karena kita ingin skor **terbesar** di depan (pasien paling kritis duluan), kita gunakan **nilai negatif**:

```python
# Skor asli: 140 (sangat kritis)
# Kita masukkan: -140
# heapq akan taruh -140 di atas (karena -140 < -95 < -20)
# Saat dikeluarkan, kita kalikan -1 lagi → dapat 140
```

Format elemen heap: `(-skor, patient_id)`  
Dengan `patient_id` sebagai tiebreaker jika skor sama.

In [ ]:
# ============================================================
# PRIORITY QUEUE IMPLEMENTATION
# Menggunakan heapq Python (min-heap)
# Trik: gunakan nilai negatif agar pasien skor tertinggi
# selalu berada di posisi terdepan (root)
# ============================================================

# Inisialisasi Priority Queue (heap) kosong
priority_queue = []   # list kosong yang akan dikelola sebagai heap


def masuk_antrian(pq, patient_id, skor):
    """
    Mendaftarkan pasien ke Priority Queue.
    
    Kompleksitas: O(log n)
    → Setelah insert, heap otomatis mengatur ulang posisi
      agar elemen dengan prioritas tertinggi tetap di root.
    
    Parameter:
        pq         : priority queue (list)
        patient_id : ID pasien (string)
        skor       : skor urgensi (int) — nilai positif asli
    """
    # Masukkan (-skor, patient_id) ke heap
    # Negatif karena heapq adalah min-heap,
    # kita ingin max-heap berdasarkan skor urgensi
    heapq.heappush(pq, (-skor, patient_id))


def layani_pasien(pq):
    """
    Mengeluarkan pasien dengan prioritas tertinggi dari antrian.
    
    Kompleksitas: O(log n)
    → Heap otomatis mengatur ulang setelah elemen root diambil.
    
    Returns:
        (skor_asli, patient_id) atau None jika antrian kosong
    """
    if not pq:  # antrian kosong
        return None
    
    skor_neg, patient_id = heapq.heappop(pq)  # ambil dari root heap
    skor_asli = -skor_neg                     # kembalikan ke nilai positif
    return skor_asli, patient_id


def lihat_antrian(pq, hash_table):
    """
    Menampilkan isi antrian saat ini (tanpa mengubah antrian).
    Heap diurutkan untuk tampilan saja.
    """
    if not pq:
        print("   (Antrian kosong)")
        return
    
    # Sort sementara untuk tampilan (tidak mengubah heap asli)
    antrian_sorted = sorted(pq)  # min-heap sort
    for urutan, (skor_neg, pid) in enumerate(antrian_sorted, 1):
        nama = hash_table[pid]["nama"]
        skor = -skor_neg
        print(f"   {urutan}. [{pid}] {nama:<10} | Skor: {skor}")


print("✅ Fungsi Priority Queue berhasil didefinisikan!")
print("   masuk_antrian()  → heapq.heappush() → O(log n)")
print("   layani_pasien()  → heapq.heappop()  → O(log n)")

---
## 🎬 9. Simulasi Sistem Antrean IGD

Sekarang kita jalankan simulasi lengkap:

**Alur kerja (sesuai slide "Alur Kerja Smart-Triage"):**
```
1. REGISTRASI  → Input ID, Hash Table menarik data profil
2. SKORING     → Input kondisi fisik, sistem hitung skor
3. ANTREAN     → Priority Queue mengatur urutan tindakan
4. PENANGANAN  → Dokter menangani pasien paling kritis
```

Pasien akan datang **satu per satu** (seperti kondisi nyata IGD), dan kita akan lihat bagaimana pasien kritis yang datang belakangan tetap mendapat prioritas.

In [ ]:
# ============================================================
# SIMULASI BAGIAN 1: PASIEN MASUK KE IGD
# Setiap pasien datang → didaftarkan ke Hash Table → dihitung
# skornya → dimasukkan ke Priority Queue
# ============================================================

# Reset struktur data untuk simulasi bersih
priority_queue = []

print("🏥 " + "=" * 58)
print("🏥  SIMULASI SISTEM ANTREAN IGD — SMART-TRIAGE")
print("🏥 " + "=" * 58)
print()
print("📍 FASE 1: PASIEN MASUK IGD")
print("-" * 60)

for pasien in data_pasien:
    pid = pasien["id"]
    nama = pasien["nama"]
    
    # Step 1: Data sudah ada di Hash Table (dari langkah sebelumnya)
    # Ambil data dari Hash Table — O(1)
    data = cari_pasien(hash_table_pasien, pid)
    
    # Step 2: Hitung skor urgensi
    skor, rincian = hitung_skor_urgensi(data)
    
    # Step 3: Masukkan ke Priority Queue — O(log n)
    masuk_antrian(priority_queue, pid, skor)
    
    # Tampilkan info kedatangan
    level = "🔴 KRITIS" if skor >= 70 else "🟡 SEDANG" if skor >= 20 else "🟢 RINGAN"
    print(f"\n  ➡️  [{pid}] {nama} tiba di IGD")
    print(f"      Kondisi : {data['kondisi']}")
    print(f"      Skor    : {skor}  {level}")
    
    # Tampilkan state antrian setelah pasien masuk
    print(f"\n      📋 Kondisi antrian sekarang:")
    lihat_antrian(priority_queue, hash_table_pasien)
    print()

In [ ]:
# ============================================================
# SIMULASI BAGIAN 2: DOKTER MELAYANI PASIEN
# Pasien dikeluarkan satu per satu dari Priority Queue
# → yang keluar pertama = skor urgensi tertinggi
# ============================================================

print("📍 FASE 2: DOKTER MEMANGGIL PASIEN")
print("-" * 60)
print()

urutan_pelayanan = []  # simpan urutan akhir untuk output final

nomor = 1
while priority_queue:  # selama masih ada pasien dalam antrian
    
    # Keluarkan pasien dengan skor tertinggi — O(log n)
    skor, patient_id = layani_pasien(priority_queue)
    
    # Akses data lengkap dari Hash Table — O(1)
    data = cari_pasien(hash_table_pasien, patient_id)
    
    level = "🔴 KRITIS" if skor >= 70 else "🟡 SEDANG" if skor >= 20 else "🟢 RINGAN"
    
    print(f"  ✅ Giliran ke-{nomor}: [{patient_id}] {data['nama']:<8} | Skor: {skor:>4}  {level}")
    print(f"             Tindakan: {data['kondisi']}")
    print()
    
    # Simpan untuk rangkuman
    urutan_pelayanan.append({
        "urutan": nomor,
        "id": patient_id,
        "nama": data["nama"],
        "skor": skor,
        "level": level
    })
    nomor += 1

print("  🏁 Semua pasien telah ditangani.")

---
## 📊 10. Output Akhir — Urutan Pelayanan Final

In [ ]:
# ============================================================
# OUTPUT AKHIR: URUTAN PELAYANAN FINAL
# Bandingkan: urutan kedatangan vs urutan pelayanan
# ============================================================

print("\n" + "=" * 65)
print("  📊 LAPORAN AKHIR: URUTAN PELAYANAN IGD SMART-TRIAGE")
print("=" * 65)

# Header tabel
print(f"\n  {'No':>3}  {'ID':<6}  {'Nama':<8}  {'Skor':>5}  {'Level':<15}  Keterangan")
print("  " + "-" * 62)

for p in urutan_pelayanan:
    # Tentukan urutan kedatangan asli
    urutan_datang = next(
        i+1 for i, d in enumerate(data_pasien) if d["id"] == p["id"]
    )
    
    # Tandai jika urutan pelayanan berbeda dari urutan datang
    naik = "⬆️ DIPRIORITASKAN" if p["urutan"] < urutan_datang else ""
    
    print(f"  {p['urutan']:>3}. {p['id']:<6}  {p['nama']:<8}  {p['skor']:>5}  {p['level']:<15}  {naik}")

print("\n" + "=" * 65)

# Ringkasan
print("\n  📌 RINGKASAN:")
print()
print("  Urutan Kedatangan  →  Urutan Dilayani")
print("  " + "-" * 40)

urutan_datang_list = [p["nama"] for p in data_pasien]
urutan_layanan_list = [p["nama"] for p in urutan_pelayanan]

for i in range(len(data_pasien)):
    tanda = "  " if urutan_datang_list[i] == urutan_layanan_list[i] else "🔁"
    print(f"  {i+1}. {urutan_datang_list[i]:<10}  →  {urutan_layanan_list[i]:<10}  {tanda}")

print()
print("  🔁 = Urutan berubah karena sistem prioritas medis")
print()

# Verifikasi: siapa yang pertama dilayani?
pertama = urutan_pelayanan[0]
print(f"  ✅ Pasien pertama dilayani : [{pertama['id']}] {pertama['nama']} (Skor: {pertama['skor']})")
print(f"  ✅ Pasien terakhir dilayani: [{urutan_pelayanan[-1]['id']}] {urutan_pelayanan[-1]['nama']} (Skor: {urutan_pelayanan[-1]['skor']})")

---
## ⚙️ 11. Perbandingan Kompleksitas Algoritma

Sesuai slide **"Kecepatan Pengurutan Data"**:

| Metode | Kompleksitas Insert | Kompleksitas Get-Max | Cocok untuk IGD? |
|---|---|---|---|
| Sorting Manual / Array | O(n log n) | O(1) setelah sort | ❌ Lambat saat real-time |
| Priority Queue (Heap) | **O(log n)** | **O(log n)** | ✅ Cepat & real-time |
| Hash Table (dict) | O(1) | O(1) | ✅ Akses data instan |

Saat ada 1.000 pasien:
- Sorting manual: ~10.000 operasi
- Priority Queue: ~10 operasi

**Perbedaan ini bisa berarti nyawa di dunia nyata.**

In [ ]:
# ============================================================
# VISUALISASI TEKS: PERBANDINGAN KOMPLEKSITAS
# ============================================================

import math

print("📊 PERBANDINGAN JUMLAH OPERASI: Heap vs Sorting Manual")
print("=" * 55)
print(f"  {'Jumlah Pasien':>15} | {'Sorting (n log n)':>18} | {'Heap (log n)':>12}")
print("  " + "-" * 52)

for n in [10, 100, 1000, 10000, 100000]:
    sorting_ops = round(n * math.log2(n))
    heap_ops    = round(math.log2(n))
    rasio       = sorting_ops // heap_ops
    print(f"  {n:>15,} | {sorting_ops:>18,} | {heap_ops:>12,}   ({rasio}x lebih lambat)")

print()
print("  ⚡ Priority Queue (Heap) jauh lebih efisien untuk sistem real-time!")

---
## 🎓 12. Kesimpulan

### Apa yang telah kita pelajari?

**1. Hash Table (`dict`) — Database Instan**
> Dengan menyimpan data pasien menggunakan ID sebagai *key*, petugas IGD bisa mengakses **semua riwayat medis dalam O(1)** — tidak peduli ada 100 atau 100.000 pasien. Tidak ada lagi pencarian berkas manual.

**2. Priority Queue (`heapq`) — Saringan Pintar**
> Setiap kali pasien baru masuk, sistem secara **otomatis menata ulang antrean** agar pasien paling kritis selalu di posisi depan. Ini berjalan dalam O(log n) — sangat cepat bahkan untuk ribuan pasien.

**3. Kombinasi Keduanya = Smart-Triage**
> - Hash Table menangani **"siapa pasien ini?"** → O(1)  
> - Priority Queue menangani **"siapa yang harus dilayani dulu?"** → O(log n)  
> - Bersama, keduanya menyelamatkan nyawa yang tidak bisa dityelamatkan oleh FIFO.

**4. Relevansi terhadap SDG 3**
> Dengan struktur data yang tepat, kita mempercepat pertolongan medis dan mengurangi angka kematian yang dapat dicegah — berkontribusi pada tujuan pembangunan berkelanjutan **"Menjamin Kehidupan yang Sehat dan Meningkatkan Kesejahteraan Seluruh Penduduk Semua Usia."**

---

### 💡 Key Takeaway

```
Masalah nyata di dunia  →  dimodelkan sebagai  →  Struktur Data
─────────────────────────────────────────────────────────────────
"Cari data pasien cepat"    →   Hash Table  (O(1) lookup)
"Siapa yang paling darurat" →   Priority Queue (O(log n) insert)
```

**Tidak perlu AI. Tidak perlu Machine Learning.  
Struktur data yang tepat sudah cukup untuk menyelamatkan nyawa.** 🏥

---
*Smart-Triage Team | Struktur Data 2024*